# LeNet on MNIST

## 학습 내용

- CNN이 필요한 이유
- Conv2d와 Feature Map
- ReLU와 MaxPool2d
- LeNet 구조 이해
- MNIST 손글씨 분류
- PyTorch로 LeNet 구현

In [9]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [10]:
# 학습 데이터 다운로드
training_data = datasets.MNIST(
    root="../data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float, scale=True)])
)

# 테스트 데이터 다운로드
test_data = datasets.MNIST(
    root="../data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float, scale=True)])
)

In [11]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

False
0
CPU


In [ ]:
batch_size = 64

training_dataloader = DataLoader(training_data, batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size)

device = "cuda" if torch.cuda.is_available() else "cpu"

for X, y in training_dataloader:
    print(f"Shape of X [N, C, H, W] {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")

    print("이동 전", X.device)

    X = X.to(device)
    y = y.to(device)

    print("이동 후", X.device)

    break

print("훈련 데이터", len(training_data))
print("테스트 데이터", len(test_data))


Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
Shape of X [N, C, H, W] torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전 cpu
이동 후 cpu
S

In [8]:
print(training_data.classes)

['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']


In [17]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Conv2d
        # 1: 입력 채널(흑백 이미지)
        # 6: 출력 채널(필터 개수 = 생성할 Feature Map 개수)
        # kernel_size=5: 5x5 필터 사용
        # padding=2: 가장자리에 2칸씩 0을 추가하여 출력 크기 유지
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        # ReLU() 활성화 함수로 음수 제거
        self.relu1 = nn.ReLU()
        # MaxPooling 2x2 영역에서 가장 큰 값 남김
        self.pool1 = nn.MaxPool2d(2)

        # 16개의 새로운 Feature Map 생성
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()  
        self.pool2 = nn.MaxPool2d(2)

        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(784, 120)
        self.relu3 = nn.ReLU()
        self.linear2 = nn.Linear(120, 84)
        self.relu4 = nn.ReLU()
        self.linear3 = nn.Linear(84, 10)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu3(x)
        x = self.linear2(x)
        x = self.relu4(x)
        x = self.linear3(x)

        return x

model = LeNet().to(device)
print(model)

LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=784, out_features=120, bias=True)
  (relu3): ReLU()
  (linear2): Linear(in_features=120, out_features=84, bias=True)
  (relu4): ReLU()
  (linear3): Linear(in_features=84, out_features=10, bias=True)
)


In [26]:
# 손실 함수
loss_fn = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [23]:
# 학습
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # forward loss 계산
        pred = model(X)
        loss = loss_fn(pred, y)

        # gradient 초기화
        optimizer.zero_grad()
        # backward + loss 계산
        loss.backward()
        # 가중치 업데이트
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")


In [24]:
# 테스트
def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y, in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            # pred.argmax(1): 샘플에서 가장 높은 점수의 인덱스를 뽑음
            # == y: 모델이 예측한 클래스랑 실제 정답 y랑 일치하면 True(1), False(0)
            # .sum().item() True(1)을 더해서 이번 배치에서 몇 개 맞았는지 뽑음
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
        
        test_loss /= num_batches
        correct /= size

    print(f"Test Error \n Accuracy : {(100 * correct):>0.1f}% \n AVG loss: {test_loss:>8f}\n")

In [27]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n------------------------")
    train_loop(training_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done")

Epoch 1
------------------------
loss: 1.303434 [    0/60000]
loss: 0.656498 [ 6400/60000]
loss: 0.496086 [12800/60000]
loss: 0.459108 [19200/60000]
loss: 0.464799 [25600/60000]
loss: 0.369291 [32000/60000]
loss: 0.284275 [38400/60000]
loss: 0.326199 [44800/60000]
loss: 0.331316 [51200/60000]
loss: 0.418401 [57600/60000]
Test Error 
 Accuracy : 90.0% 
 AVG loss: 0.308725

Epoch 2
------------------------
loss: 0.301409 [    0/60000]
loss: 0.223006 [ 6400/60000]
loss: 0.183602 [12800/60000]
loss: 0.304315 [19200/60000]
loss: 0.211844 [25600/60000]
loss: 0.237126 [32000/60000]
loss: 0.123973 [38400/60000]
loss: 0.278193 [44800/60000]
loss: 0.222211 [51200/60000]
loss: 0.319111 [57600/60000]
Test Error 
 Accuracy : 93.7% 
 AVG loss: 0.195929

Epoch 3
------------------------
loss: 0.144938 [    0/60000]
loss: 0.171538 [ 6400/60000]
loss: 0.149378 [12800/60000]
loss: 0.249304 [19200/60000]
loss: 0.122158 [25600/60000]
loss: 0.184761 [32000/60000]
loss: 0.073487 [38400/60000]
loss: 0.250989

In [28]:
# 모델 저장
torch.save(model.state_dict(), "../models/lenet_mnist.pth")
print("Save PyTorch Model State to lenet_mnist.pth")

Save PyTorch Model State to lenet_mnist.pth


## 모델 구조

LeNet-5 기반 CNN:
- Conv2d(1, 6, kernel_size=5, padding=2) -> ReLU -> MaxPool2d(2)
- Conv2d(6, 16, kernel_size=5, padding=2) -> ReLU -> MaxPool2d(2)
- Flatten -> Linear(784, 120) -> ReLU -> Linear(120, 84) -> ReLU -> Linear(84, 10)

각 단계별 텐서 shape 변화:
- 입력: (1, 28, 28)
- conv1 + pool1 이후: (6, 14, 14)
- conv2 + pool2 이후: (16, 7, 7)
- flatten 이후: 784(16 * 7 * 7)

## 실험 - Learning Rate의 영향

| lr | Optimizer | Epochs | Accuracy |
|------|-----------|--------|----------|
| 0.001 | SGD | 10 | 63.3% |
| 0.01 | SGD | 10 | 97.7% |

- SGD는 momentum이 없어서 lr이 너무 적으면 loss landscape의 평탄한 구간(plateau)에서 거의 움직이지 못한다.
- lr=0.001일 때 8 epoch까지 loss가 거의 줄어들지 않다가 마지막에 급격히 떨어짐(학습이 너무 늦게 시작됨)
- lr=0.01로 키우니까 1 epoch만에 정확도가 90%로 상승

## 결과
- 최종 테스트 정확도 : 97.7%
- 모델 저장: `models/lenet_mnist.pth`

## 배운 점
- MaxPool2d(2)는 2x2 윈도우에서 최댓값만 남겨 가로/세로를 절반으로 줄인다.
- Conv 레이어가 깊어질수록 채널 수를 늘려 더 복잡한 특징을 학습하도록 설계하는 것이 일반적이다.
- learning rate는 모델 성능에 결정적 영향을 미치는 하이퍼 파라미터
